In [10]:
# torch, faiss (k-NN/perplexity graph construction) and PHATE's numba-based internals
# all get loaded into one process here; with unconstrained thread pools that combination
# makes macOS abort with a duplicate-OpenMP-runtime error (kernel crash / segfault).
# Capping thread counts before any of those libraries import, plus the (unsafe but
# standard) escape hatch as a second line of defense, avoids it.
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')

import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path

from plot_utils_era5 import *

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/ERA5/scripts/")
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess
import phate
import anndata as ad
import concord as ccd
import synthia as syn
from scipy.interpolate import LinearNDInterpolator



def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    if model_type in ['random_crop', 'regular']:
        model = KernelNN(
            width=config['width'],
            ker_width=config['ker_width'],
            depth=config['depth'],
            ker_in=config['edge_attr'],
            in_width=config['node_features'],
            out_width=config['out_width']
        ).to(device)
        
    elif model_type in ['subgraph', 'random_crop_subgraph']:
        model = KernelNN_subgraph(
            width=config['width'],  # number features per node after lifting
            ker_width=config['ker_width'],  # dimension of single kernel layer
            depth=config['depth'],  # number of graph layers / message passings
            ker_in=config['edge_attr'],  # number of edge features
            known_node_dim=config['input_node_features'],  # only needed when lifting is applied within model
            unknown_node_dim=config['output_node_features'],
            out_width=config['out_width']
        ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config

Using device: mps


In [11]:
# No trained-model weights available -- config only, NOFE is not computed in this notebook.
config = {
    'model_config_path': 'ERA5/configurations/subgraph/model_configs/base2_d2.json',
    'dataset_id': 'sub_fixcrop_median_woPos_tkernel',
    'dataset_path': 'ERA5/datasets/subgraph',
    'learning_rate': 0.001,
    'scheduler_step': 2,
    'scheduler_gamma': 0.5,
    'epochs': 25,
    'random_seed': 42,
    'use_gradient_clipping': True,
    'track_gradients': True,
    'checkpoint_every': 5,
    'width': 64,
    'ker_width': 64,
    'depth': 2,
    'input_node_features': 11,
    'output_node_features': 11,
    'edge_attr': 7,
    'out_width': 3,
    'data_folder_path': '../data/',
    'dataset_save_path': '../datasets/subgraph/',
    'lat_min': 30,
    'lat_max': 70,
    'lon_min': -10,
    'lon_max': 30,
    'N_in_min': 100,
    'N_in_max': 5000,
    'N_out_min': 1000,
    'N_out_max': 10000,
    'k_in': 5,
    'k_out': 5,
    'k_cross': 5,
    'perplexity': 20,
    'input_sampling_method': 'random',
    'output_sampling_method': 'random',
    'train_date_start_end': ['2018-01-01', '2018-12-31'],
    'test_date_start_end': ['2019-01-01', '2019-01-31'],
    'train_samples_per_date': 10,
    'test_samples_per_date': 2,
    'feature_mode': 'channel_only',
    'interpolate_graph_features': True,
    'sigma_method': 'tkernel',
    'vars': ['w', 'v', 'z', 'q', 'r', 't', 'd', 'o3', 'pv', 'u', 'vo'],
}

In [12]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k_in = config["k_in"]
k_out = config["k_out"]
k_cross = config["k_cross"]
perplexity = config["perplexity"]
input_sampling_method = config["input_sampling_method"]
output_sampling_method = config["output_sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]
interpolate_graph_features = config["interpolate_graph_features"]

In [13]:
dates = pd.date_range("2019-01-01", "2019-01-31").strftime("%Y-%m-%d").tolist()

N_in = 200
N_out = 1000

input_sampling_method = "random"
output_sampling_method = "random"
bbox = lon_min, lat_min, lon_max, lat_max

Generator = Data_Generator(data_path)


print('generate data')

graphs=[]
locations_list=[]
X_list=[]

for date in dates:
    Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
    in_graph = Generator.sample_graph(N=N_in, k=k_in, perplexity=perplexity, sampling_method=input_sampling_method, gen_feature_mode=feature_mode)
    #full_data = Generator.sample_graph(N=100000, k=0, perplexity=0, sampling_method='random', gen_feature_mode=feature_mode)
    out_graph = Generator.sample_graph(N=N_out, k=k_out, perplexity=perplexity, sampling_method=output_sampling_method, gen_feature_mode=feature_mode)
    cross_graph = Generator.connect_graphs(in_graph=in_graph, out_graph=out_graph, k_cross=k_cross)
    cross_graph["x"] = Generator.construct_output_node_features(cross_graph=cross_graph, interpolate=interpolate_graph_features, gen_feature_mode=feature_mode)
    sample = cross_graph.to(device)
    graphs.append(sample)

generate data


In [14]:
#pred_nofe_list = []
#pred_pca_list = []
#pred_tsne_list = []
#pred_umap_list = []
pred_phate_list = []
pred_spatialPCA_list = []
pred_concord_list = []
pred_fpca_list = []

# Start the spatialPCA R process ONCE, outside the loop -- library(SpatialPCA) (Seurat,
# SPARK, etc.) is the slow part to load, and conda run's own activation overhead also
# used to get paid on every single call. It now stays alive and answers one request per
# line over stdin/stdout for the whole loop below.
spdir = f'{project_root}/ERA5/spatialpca'
# Unique per process so concurrently running scripts/notebooks don't collide on the
# same buffer filenames (a fixed call_id caused FileNotFoundError / silent data
# corruption when two spatialPCA calls raced on the same X.csv/coords.csv paths).
spatialpca_call_id = f'spatialpca_buffer_{os.getpid()}'
spatialpca_proc = subprocess.Popen(
    ['conda', 'run', '-n', 'spatialpca', '--no-capture-output', 'Rscript', 'execute_spatialpca_server.R'],
    cwd=spdir, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, bufsize=1,
)
ready = spatialpca_proc.stdout.readline().strip()
if ready != 'READY':
    raise RuntimeError(f'spatialPCA server failed to start: {ready!r}\n{spatialpca_proc.stderr.read()}')
print('spatialPCA server ready.')


def spatialpca_embed(X, coords, call_id, n_components=3):
    np.savetxt(f'{spdir}/{call_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{spdir}/{call_id}_coords.csv', coords, delimiter=',')

    spatialpca_proc.stdin.write(f'{call_id} {n_components}\n')
    spatialpca_proc.stdin.flush()
    response = spatialpca_proc.stdout.readline().strip()

    os.remove(f'{spdir}/{call_id}_X.csv')
    os.remove(f'{spdir}/{call_id}_coords.csv')

    if response != 'OK':
        raise RuntimeError(f'spatialPCA failed for {call_id}: {response}')

    Y = np.loadtxt(f'{spdir}/res_{call_id}.csv', delimiter=',', skiprows=1)
    os.remove(f'{spdir}/res_{call_id}.csv')
    return Y


def concord_embed(X, n_components=3):
    adata = ad.AnnData(X=X.astype(np.float32))
    cur_ccd = ccd.Concord(
        adata=adata, input_feature=list(adata.var_names), device=device,
        latent_dim=n_components, verbose=False,
        # preload_dense=True: concord-sc has a bug where the num_workers kwarg is
        # accepted but never forwarded to its internal DataLoaderManager, so passing
        # num_workers=0 alone does NOT stop it from spawning ~4 worker subprocesses per
        # DataLoader. preload_dense=True hits a different code path in
        # concord/model/dataloader.py that correctly forces num_workers to 0 -- without
        # this, every worker spawn re-imports torch/anndata/concord from scratch, which
        # is fine in a notebook (Jupyter's kernel is __main__, not this file) but turns
        # a single call into tens of seconds of pure import overhead when exported/run
        # as a plain script.
        preload_dense=True,
    )
    cur_ccd.fit_transform(output_key='Concord')
    return adata.obsm['Concord']


def fpca_embed(X, n_components=3):
    generator = syn.FPCADataGenerator()
    generator.fit(X.astype(np.float64), n_fpca_components=n_components)
    return generator.eig_scores


for i, sample in enumerate(graphs):
    print(f'{i}/{len(graphs)}')
    
    x_in = np.array(sample.source.meta.gps)
    x_out = np.array(sample.target.meta.gps)
    
    X = sample.source.normed_values.detach().cpu().numpy()

    """
    # NOFE
    print('NOFE ...', end='\r')
    pred_nofe = model(sample).detach().cpu().numpy()
    pred_nofe_list.append(pred_nofe)

    # PCA
    print('PCA ...', end='\r')
    pca = PCA(n_components=3)
    pred_pca = pca.fit_transform(X)
    interp = LinearNDInterpolator(x_in, pred_pca)
    z_out = interp(x_out)
    mask = np.isnan(z_out)
    z_out[mask] = 0
    pred_pca_list.append(z_out)

    # t-SNE
    print('t-SNE ...', end='\r')
    pred_tsne = TSNE(
        n_components=3,
        perplexity=30,
        learning_rate="auto",
        random_state=42
    ).fit_transform(X)
    interp = LinearNDInterpolator(x_in, pred_tsne)
    z_out = interp(x_out)
    mask = np.isnan(z_out)
    z_out[mask] = 0
    pred_tsne_list.append(z_out)

    # UMAP
    print('UMAP ...', end='\r')
    umap_id = 'umap_buffer'
    dir = f'{project_root}/ERA5/umap'
    np.save(f'{dir}/{umap_id}.npy', X)
    subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id} > /dev/null 2>&1", shell=True)
    pred_umap = np.load(f'{dir}/res_{umap_id}.npy')
    interp = LinearNDInterpolator(x_in, pred_umap)
    z_out = interp(x_out)
    mask = np.isnan(z_out)
    z_out[mask] = 0
    pred_umap_list.append(z_out)

    os.remove(f'{dir}/{umap_id}.npy')
    os.remove(f'{dir}/res_{umap_id}.npy')
    """

    # PHATE
    print('PHATE ...', end='\r')
    pred_phate = phate.PHATE(
        n_components=3, knn=15, mds_solver='smacof', random_state=42, verbose=False
    ).fit_transform(X)
    interp = LinearNDInterpolator(x_in, pred_phate)
    z_out = interp(x_out)
    mask = np.isnan(z_out)
    z_out[mask] = 0
    pred_phate_list.append(z_out)

    # spatialPCA
    print('spatialPCA ...', end='\r')
    coords_in = sample.source.normed_coords.detach().cpu().numpy()
    pred_spatialPCA = spatialpca_embed(X, coords_in, spatialpca_call_id, n_components=3)
    interp = LinearNDInterpolator(x_in, pred_spatialPCA)
    z_out = interp(x_out)
    mask = np.isnan(z_out)
    z_out[mask] = 0
    pred_spatialPCA_list.append(z_out)

    # Concord
    print('Concord ...', end='\r')
    pred_concord = concord_embed(X, n_components=3)
    interp = LinearNDInterpolator(x_in, pred_concord)
    z_out = interp(x_out)
    mask = np.isnan(z_out)
    z_out[mask] = 0
    pred_concord_list.append(z_out)

    # functionalPCA
    print('fPCA ...', end='\r')
    pred_fpca = fpca_embed(X, n_components=3)
    interp = LinearNDInterpolator(x_in, pred_fpca)
    z_out = interp(x_out)
    mask = np.isnan(z_out)
    z_out[mask] = 0
    pred_fpca_list.append(z_out)


spatialpca_proc.stdin.write('QUIT\n')
spatialpca_proc.stdin.flush()
spatialpca_proc.wait(timeout=10)

spatialPCA server ready.
0/31


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/siche9897/miniconda3/envs/neural

concord - WARNING - domain/batch information not found, all samples will be treated as from single domain/batch.
concord - WARNING - Only one domain found in the data. Setting p_intra_domain to 1.0.
Epoch 0 Training
Epoch 1 Training
Epoch 2 Training
Epoch 3 Training
Epoch 4 Training
Epoch 5 Training
Epoch 6 Training
Epoch 7 Training
Epoch 8 Training


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 9 Training
Epoch 10 Training
Epoch 11 Training
Epoch 12 Training
Epoch 13 Training
Epoch 14 Training
1/31 ...


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in

... (117 outputs left out)

/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 8 Training
Epoch 9 Training
Epoch 10 Training
Epoch 11 Training
Epoch 12 Training
Epoch 13 Training
Epoch 14 Training


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


0

In [15]:
def pairwise_distances(X):
    # X: (N, d)
    norms = np.sum(X**2, axis=1, keepdims=True)  # (N, 1)
    
    dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)
    
    # numerical stability: avoid tiny negatives
    dist2 = np.maximum(dist2, 0.0)
    
    return np.sqrt(dist2)

def distance_preservation(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)
    
    # Take upper triangle (avoid duplicates + diagonal)
    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]
    
    # Correlation (1 = perfect preservation)
    corr = np.corrcoef(dH, dL)[0, 1]
    
    return corr

def stress(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)
    
    # Take upper triangle (avoid duplicates + diagonal)
    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]
    
    # stress 
    stress = np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))
    
    return stress

In [16]:

def edge_distances(X, edge_indices):
    # edge_indices is PyG's [2, E] layout (row 0 = source nodes, row 1 = target
    # nodes) -- NOT [E, 2]. Indexing it as edge_indices[:, 0]/[:, 1] only pulls out
    # 2 elements total (one column) instead of the E-length source/target arrays,
    # which silently makes corr (local) meaningless (trivially +-1, or NaN if those
    # 2 values happen to coincide, e.g. two self-loop edges both at distance 0).
    diff = X[edge_indices[0]] - X[edge_indices[1]]
    return np.linalg.norm(diff, axis=1)

def edge_distance_preservation(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    
    # Correlation
    corr = np.corrcoef(dH, dL)[0, 1]
    return corr

def edge_stress(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    
    return np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))

#def weighted_edge_stress(H, L, edge_indices, weights):
#    dH = edge_distances(H, edge_indices)
#    dL = edge_distances(L, edge_indices)   
#    return np.sqrt(np.sum(weights * (dH - dL)**2) / np.sum(weights * dH**2))

In [17]:
import numpy as np
import pandas as pd

# Accumulate scores across all graphs
#nofe_preservation = []
#nofe_stress = []
#nofe_edge_preservation = []
#nofe_edge_stress = []

#pca_preservation = []
#pca_stress = []
#pca_edge_preservation = []
#pca_edge_stress = []

#tsne_preservation = []
#tsne_stress = []
#tsne_edge_preservation = []
#tsne_edge_stress = []

#umap_preservation = []
#umap_stress = []
#umap_edge_preservation = []
#umap_edge_stress = []

phate_preservation = []
phate_stress = []
phate_edge_preservation = []
phate_edge_stress = []

spatialpca_preservation = []
spatialpca_stress = []
spatialpca_edge_preservation = []
spatialpca_edge_stress = []

concord_preservation = []
concord_stress = []
concord_edge_preservation = []
concord_edge_stress = []

fpca_preservation = []
fpca_stress = []
fpca_edge_preservation = []
fpca_edge_stress = []


for i in range(len(graphs)):

    edge_indices = graphs[i].target.edge_index.detach().cpu().numpy()

    H = graphs[i].target.meta.x_true.detach().cpu().numpy()

    #Lnofe = pred_nofe_list[i]
    #Lpca  = pred_pca_list[i]
    #Ltsne = pred_tsne_list[i]
    #Lumap = pred_umap_list[i]
    Lphate = pred_phate_list[i]
    LspatialPCA = pred_spatialPCA_list[i]
    Lconcord = pred_concord_list[i]
    Lfpca = pred_fpca_list[i]

    """
    # NOFE
    nofe_preservation.append(distance_preservation(H, Lnofe))
    nofe_stress.append(stress(H, Lnofe))
    nofe_edge_preservation.append(
        edge_distance_preservation(H, Lnofe, edge_indices)
    )
    nofe_edge_stress.append(
        edge_stress(H, Lnofe, edge_indices)
    )

    # PCA
    pca_preservation.append(distance_preservation(H, Lpca))
    pca_stress.append(stress(H, Lpca))
    pca_edge_preservation.append(
        edge_distance_preservation(H, Lpca, edge_indices)
    )
    pca_edge_stress.append(
        edge_stress(H, Lpca, edge_indices)
    )

    # t-SNE
    tsne_preservation.append(distance_preservation(H, Ltsne))
    tsne_stress.append(stress(H, Ltsne))
    tsne_edge_preservation.append(
        edge_distance_preservation(H, Ltsne, edge_indices)
    )
    tsne_edge_stress.append(
        edge_stress(H, Ltsne, edge_indices)
    )

    # UMAP
    umap_preservation.append(distance_preservation(H, Lumap))
    umap_stress.append(stress(H, Lumap))   # fixed typo here
    umap_edge_preservation.append(
        edge_distance_preservation(H, Lumap, edge_indices)
    )
    umap_edge_stress.append(
        edge_stress(H, Lumap, edge_indices)
    )
    """

    # PHATE
    phate_preservation.append(distance_preservation(H, Lphate))
    phate_stress.append(stress(H, Lphate))
    phate_edge_preservation.append(
        edge_distance_preservation(H, Lphate, edge_indices)
    )
    phate_edge_stress.append(
        edge_stress(H, Lphate, edge_indices)
    )

    # spatialPCA
    spatialpca_preservation.append(distance_preservation(H, LspatialPCA))
    spatialpca_stress.append(stress(H, LspatialPCA))
    spatialpca_edge_preservation.append(
        edge_distance_preservation(H, LspatialPCA, edge_indices)
    )
    spatialpca_edge_stress.append(
        edge_stress(H, LspatialPCA, edge_indices)
    )

    # Concord
    concord_preservation.append(distance_preservation(H, Lconcord))
    concord_stress.append(stress(H, Lconcord))
    concord_edge_preservation.append(
        edge_distance_preservation(H, Lconcord, edge_indices)
    )
    concord_edge_stress.append(
        edge_stress(H, Lconcord, edge_indices)
    )

    # functionalPCA
    fpca_preservation.append(distance_preservation(H, Lfpca))
    fpca_stress.append(stress(H, Lfpca))
    fpca_edge_preservation.append(
        edge_distance_preservation(H, Lfpca, edge_indices)
    )
    fpca_edge_stress.append(
        edge_stress(H, Lfpca, edge_indices)
    )


def mean_std(arr):
    return f"{np.mean(arr):.4f} ± {np.std(arr):.4f}"


scores = pd.DataFrame(
    {
        #"nofe": [
        #    mean_std(nofe_preservation),
        #    mean_std(nofe_stress),
        #    mean_std(nofe_edge_preservation),
        #    mean_std(nofe_edge_stress),
        #],
        #"pca": [
        #    mean_std(pca_preservation),
        #    mean_std(pca_stress),
        #    mean_std(pca_edge_preservation),
        #    mean_std(pca_edge_stress),
        #],
        #"tsne": [
        #    mean_std(tsne_preservation),
        #    mean_std(tsne_stress),
        #    mean_std(tsne_edge_preservation),
        #    mean_std(tsne_edge_stress),
        #],
        #"umap": [
        #    mean_std(umap_preservation),
        #    mean_std(umap_stress),
        #    mean_std(umap_edge_preservation),
        #    mean_std(umap_edge_stress),
        #],
        "phate": [
            mean_std(phate_preservation),
            mean_std(phate_stress),
            mean_std(phate_edge_preservation),
            mean_std(phate_edge_stress),
        ],
        "spatialpca": [
            mean_std(spatialpca_preservation),
            mean_std(spatialpca_stress),
            mean_std(spatialpca_edge_preservation),
            mean_std(spatialpca_edge_stress),
        ],
        "concord": [
            mean_std(concord_preservation),
            mean_std(concord_stress),
            mean_std(concord_edge_preservation),
            mean_std(concord_edge_stress),
        ],
        "fpca": [
            mean_std(fpca_preservation),
            mean_std(fpca_stress),
            mean_std(fpca_edge_preservation),
            mean_std(fpca_edge_stress),
        ],
    },
    index=[
        "corr (global)",
        "stress (global)",
        "corr (local)",
        "stress (local)",
    ],
)

print(scores)

/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_42884/373983649.py:5: RuntimeWarning: divide by zero encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)
/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_42884/373983649.py:5: RuntimeWarning: overflow encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)
/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_42884/373983649.py:5: RuntimeWarning: invalid value encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)


                           phate       spatialpca          concord  \
corr (global)    0.4015 ± 0.0771  0.4801 ± 0.0687  0.2702 ± 0.1076   
stress (global)  0.9834 ± 0.0008  0.4282 ± 0.0253  0.8730 ± 0.0323   
corr (local)     0.1724 ± 0.0620  0.2620 ± 0.0573  0.1069 ± 0.0486   
stress (local)   0.9953 ± 0.0007  0.7503 ± 0.0250  0.9474 ± 0.0134   

                            fpca  
corr (global)    0.5793 ± 0.0572  
stress (global)  0.4428 ± 0.0297  
corr (local)     0.3399 ± 0.0682  
stress (local)   0.7664 ± 0.0247  


In [18]:
print(scores)

                           phate       spatialpca          concord  \
corr (global)    0.4015 ± 0.0771  0.4801 ± 0.0687  0.2702 ± 0.1076   
stress (global)  0.9834 ± 0.0008  0.4282 ± 0.0253  0.8730 ± 0.0323   
corr (local)     0.1724 ± 0.0620  0.2620 ± 0.0573  0.1069 ± 0.0486   
stress (local)   0.9953 ± 0.0007  0.7503 ± 0.0250  0.9474 ± 0.0134   

                            fpca  
corr (global)    0.5793 ± 0.0572  
stress (global)  0.4428 ± 0.0297  
corr (local)     0.3399 ± 0.0682  
stress (local)   0.7664 ± 0.0247  
